# ETAPA 1: Pipeline + modelos simples (sem Optuna)

In [ ]:
!pip -q install optuna xgboost
# TabPFN
!pip -q install tabpfn


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor

RANDOM_STATE = 42

In [ ]:
# ===== (2) Definir X, y e groups =====
# Ajuste os nomes caso estejam diferentes:
TARGET_COL = "R_pressure"
TIME_COL   = "Q_time"
GROUP_COL  = "curve_id"   # importante para CV sem vazamento

# Se tiver colunas auxiliares como t_index, pode manter em X ou tirar.
X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL].astype(float)

X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL].astype(float)

groups = train_df[GROUP_COL].values  # para GroupKFold

# Colunas numéricas/categóricas (provavelmente tudo numérico)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

num_cols[:10], cat_cols


In [ ]:
# ===== (3) Pré-processamento (igual para todos) =====
pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            # se tiver categoria, eu coloco OneHot; se não tiver, tudo bem
        ]), cat_cols),
    ],
    remainder="drop"
)

cv = GroupKFold(n_splits=5)

def eval_cv_rmse(pipe):
    # cross_val_score usa "neg_root_mean_squared_error"
    scores = cross_val_score(
        pipe, X_train, y_train,
        cv=cv, groups=groups,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    rmse = -scores.mean()
    return float(rmse)


In [ ]:
train_df.shape
train_df.columns

Removendo colunas que não serão usadas como features

In [ ]:
TARGET_COL = "R_pressure"

drop_cols = ["R_pressure", "curve_id", "t_index"]

X = train_df.drop(columns=drop_cols)
y = train_df[TARGET_COL].astype(float)


Fazendo amostragem inteligente

In [ ]:
SAMPLE_N = 30000

sample_idx = np.random.RandomState(42).choice(
    len(X), size=min(SAMPLE_N, len(X)), replace=False
)

X_sample = X.iloc[sample_idx]
y_sample = y.iloc[sample_idx]

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_va, y_tr, y_va = train_test_split(
    X_sample, y_sample,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
# Import necessary components for ColumnTransformer and pre-processing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import numpy as np # Import numpy for np.number if not already imported

# --- Start of fix ---
# Recalculate num_cols and cat_cols based on X_tr, as X_tr has already had columns dropped.
# This ensures the ColumnTransformer only tries to process columns present in the actual data.
num_cols_for_preprocessor = X_tr.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_for_preprocessor = X_tr.select_dtypes(exclude=[np.number]).columns.tolist()

# Re-define the preprocessor with the correct columns for X_tr
pre_fixed = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols_for_preprocessor),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            # se tiver categoria, eu coloco OneHot; se não tiver, tudo bem
        ]), cat_cols_for_preprocessor),
    ],
    remainder="drop"
)
# --- End of fix ---


models_fast = {

    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=120,
        random_state=42,
        n_jobs=-1
    ),

    "XGB_light": XGBRegressor(
        n_estimators=120,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        n_jobs=-1,
        random_state=42
    ),

    "Ridge": Ridge(alpha=1.0)
}

baseline_results = []

for name, model in models_fast.items():
    # Use pre_fixed instead of pre
    pipe = Pipeline([("pre", pre_fixed), ("model", model)])
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_va)
    rmse = np.sqrt(((y_va - pred) ** 2).mean())
    baseline_results.append((name, rmse))
    print(name, "RMSE:", rmse)

baseline_results

Optuna (10 trials) em amostra

In [ ]:
!pip -q install optuna xgboost

import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

RANDOM_STATE = 42
TARGET_COL = "R_pressure"

# ========= 1) Preparar X e y (sem colunas que não podem virar feature) =========
X_full = train_df.drop(columns=[TARGET_COL, "curve_id", "t_index"], errors="ignore")
y_full = train_df[TARGET_COL].astype(float)

# ========= 2) Amostra =========
SAMPLE_N = 30000
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    len(X_full), size=min(SAMPLE_N, len(X_full)), replace=False
)

X = X_full.iloc[sample_idx].copy()
y = y_full.iloc[sample_idx].copy()

# Split interno (para CV mais rápido e consistente)
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# ========= 3) Preprocessamento =========
num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_tr.select_dtypes(exclude=[np.number]).columns.tolist()

pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        # Se tiver categorias (provavelmente não), a gente adiciona depois.
    ],
    remainder="drop"
)

# ========= 4) CV leve =========
cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def eval_cv_rmse(pipe):
    scores = cross_val_score(
        pipe, X_tr, y_tr,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    return float(-scores.mean())

N_TRIALS = 10  # pedido do professor

# ========= 5) Optuna: ExtraTrees =========
def objective_extratrees(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=100),
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, None]),
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }
    model = ExtraTreesRegressor(**params)
    pipe = Pipeline([("pre", pre), ("model", model)])
    return eval_cv_rmse(pipe)

study_et = optuna.create_study(direction="minimize", study_name="ExtraTrees_RMSE")
study_et.optimize(objective_extratrees, n_trials=N_TRIALS, show_progress_bar=True)

# ========= 6) Optuna: XGBoost =========
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 5.0),
        "tree_method": "hist",
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }
    model = XGBRegressor(**params)
    pipe = Pipeline([("pre", pre), ("model", model)])
    return eval_cv_rmse(pipe)

study_xgb = optuna.create_study(direction="minimize", study_name="XGB_RMSE")
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)

# ========= 7) Optuna: MLP (Rede Neural) =========
def objective_mlp(trial):
    n_layers = trial.suggest_int("n_hidden_layers", 1, 3)
    hidden_layers = tuple(
        trial.suggest_int(f"n_units_l{i+1}", 16, 256, step=16)
        for i in range(n_layers)
    )
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])
    alpha = trial.suggest_float("alpha", 1e-6, 1e-2, log=True)
    lr_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)

    model = MLPRegressor(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        alpha=alpha,
        learning_rate_init=lr_init,
        max_iter=200,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=RANDOM_STATE
    )
    pipe = Pipeline([("pre", pre), ("model", model)])
    return eval_cv_rmse(pipe)

study_mlp = optuna.create_study(direction="minimize", study_name="MLP_RMSE")
study_mlp.optimize(objective_mlp, n_trials=N_TRIALS, show_progress_bar=True)

# ========= 8) Ranking final =========
results = [
    ("ExtraTrees", study_et.best_value, study_et.best_params),
    ("XGBoost", study_xgb.best_value, study_xgb.best_params),
    ("MLP", study_mlp.best_value, study_mlp.best_params),
]

results_sorted = sorted(results, key=lambda x: x[1])

print("\n===== RESULTADOS (RMSE CV - menor é melhor) =====")
for name, rmse, params in results_sorted:
    print(f"{name}: RMSE={rmse:.6f}")
    print("  params:", params)
    print("-" * 60)

Melhor modelo XGBoost

In [ ]:
# =========================
# 0) Imports e configs
# =========================
!pip -q install xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
TARGET_COL = "R_pressure"
TIME_COL   = "Q_time"
CURVE_COL  = "curve_id"

In [ ]:
# =========================
# 1) Carregar dados
# =========================
train_path = "/content/train_timeseries.xlsx"
test_path  = "/content/test_timeseries.xlsx"

train_df = pd.read_excel(train_path)
test_df  = pd.read_excel(test_path)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
train_df.head()

In [ ]:
# =========================
# 2) Preparar X/y (treino e teste completos)
# =========================
drop_cols = [TARGET_COL, CURVE_COL, "t_index"]  # t_index é opcional

X_train_full = train_df.drop(columns=drop_cols, errors="ignore")
y_train_full = train_df[TARGET_COL].astype(float)

X_test_full  = test_df.drop(columns=drop_cols, errors="ignore")
y_test_full  = test_df[TARGET_COL].astype(float)

print("X_train_full:", X_train_full.shape, "| y_train_full:", y_train_full.shape)
print("X_test_full :", X_test_full.shape,  "| y_test_full :", y_test_full.shape)

In [ ]:
# =========================
# 3) Pré-processamento (recalcular num_cols sempre!)
# =========================
num_cols = X_train_full.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train_full.select_dtypes(exclude=[np.number]).columns.tolist()

pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
    ],
    remainder="drop"
)

print("Num cols:", len(num_cols))
print("Cat cols:", len(cat_cols))

In [ ]:
best_params = {
    "n_estimators": 800,
    "max_depth": 9,
    "learning_rate": 0.16920306292293242,
    "subsample": 0.9819393281062181,
    "colsample_bytree": 0.9288186113582297,
    "min_child_weight": 7.029942756969786,
    # se tiver reg_alpha/reg_lambda no seu print, pode adicionar aqui
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": RANDOM_STATE
}

best_xgb = XGBRegressor(**best_params)

pipe_best = Pipeline([
    ("pre", pre),
    ("model", best_xgb)
])

# Treinar
pipe_best.fit(X_train_full, y_train_full)

# Prever
y_pred = pipe_best.predict(X_test_full)

print("Pred shape:", y_pred.shape)

In [ ]:
mse  = mean_squared_error(y_test_full, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_full, y_pred)
r2   = r2_score(y_test_full, y_pred)

print("=== MÉTRICAS NO TESTE (curva inteira) ===")
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)

In [ ]:
curve_example = test_df[CURVE_COL].iloc[0]  # primeira curva do teste
mask = (test_df[CURVE_COL] == curve_example)

t = test_df.loc[mask, TIME_COL].values
y_real = y_test_full.loc[mask].values
y_hat  = y_pred[mask.values]

# ordenar por tempo para o gráfico ficar certo
order = np.argsort(t)

plt.figure()
plt.plot(t[order], y_real[order], label="Real")
plt.plot(t[order], y_hat[order], label="Predito")
plt.xlabel("Tempo (Q_time)")
plt.ylabel("Pressão (R_pressure)")
plt.title(f"Curva de pressão — curve_id={curve_example}")
plt.legend()
plt.show()

Curva R²

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# Ordenar teste por tempo
order_global = np.argsort(test_df["Q_time"].values)

y_real_ord = y_test_full.values[order_global]
y_pred_ord = y_pred[order_global]

r2_progressivo = []

for i in range(50, len(y_real_ord), 200):  # começa com 50 pontos
    r2_progressivo.append(
        r2_score(y_real_ord[:i], y_pred_ord[:i])
    )

plt.figure()
plt.plot(r2_progressivo)
plt.xlabel("Blocos de tempo acumulados")
plt.ylabel("R² acumulado")
plt.title("Evolução do R² ao longo da curva")
plt.show()


Calculando o erro


In [ ]:
erro = y_test_full.values - y_pred

In [ ]:
import matplotlib.pyplot as plt

# Criar DataFrame com erro
analysis_df = X_test_full.copy()
analysis_df["erro"] = erro

# Plotar erro vs cada variável numérica
for col in num_cols:
    plt.figure()
    plt.scatter(analysis_df[col], analysis_df["erro"], s=5)
    plt.xlabel(col)
    plt.ylabel("Erro (Real - Predito)")
    plt.title(f"Erro vs {col}")
    plt.show()

Rodando o TabPFNRegressor

In [ ]:
!pip -q install tabpfn

Usando 1000 para rodar melhor TabPFN com 1000 amostras (CPU)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42
TARGET_COL = "R_pressure"

# X e y do treino completo (reaproveita seu X_train_full / y_train_full se já existir)
X_all = train_df.drop(columns=[TARGET_COL, "curve_id", "t_index"], errors="ignore")
y_all = train_df[TARGET_COL].astype(float)

SAMPLE_N = 10000
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    len(X_all), size=min(SAMPLE_N, len(X_all)), replace=False
)

X_s = X_all.iloc[sample_idx].copy()
y_s = y_all.iloc[sample_idx].copy()

X_tr, X_va, y_tr, y_va = train_test_split(
    X_s, y_s, test_size=0.2, random_state=RANDOM_STATE
)

In [ ]:
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy="median")
X_tr_imp = imp.fit_transform(X_tr)
X_va_imp = imp.transform(X_va)

In [ ]:
from huggingface_hub import login
login()

In [ ]:
SAMPLE_N = 1000
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    len(X_all), size=min(SAMPLE_N, len(X_all)), replace=False
)

X_s = X_all.iloc[sample_idx].copy()
y_s = y_all.iloc[sample_idx].copy()

X_tr, X_va, y_tr, y_va = train_test_split(
    X_s, y_s, test_size=0.2, random_state=RANDOM_STATE
)

imp = SimpleImputer(strategy="median")
X_tr_imp = imp.fit_transform(X_tr)
X_va_imp = imp.transform(X_va)

from tabpfn import TabPFNRegressor
tab_model = TabPFNRegressor(device="cpu")
tab_model.fit(X_tr_imp, y_tr)
y_hat = tab_model.predict(X_va_imp)

rmse = np.sqrt(mean_squared_error(y_va, y_hat))
r2   = r2_score(y_va, y_hat)

print("TABPFN - RMSE:", rmse)
print("TABPFN - R2  :", r2)

In [ ]:
# ===========================
# BLOCO FINAL - COLAB
# ===========================

# ---- (0) Instalações ----
!pip -q install xgboost tabpfn huggingface_hub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET_COL = "R_pressure"
TIME_COL   = "Q_time"
CURVE_COL  = "curve_id"

# ---- (1) Carregar dados ----
train_path = "/content/train_timeseries.xlsx"
test_path  = "/content/test_timeseries.xlsx"

train_df = pd.read_excel(train_path)
test_df  = pd.read_excel(test_path)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

# ---- (2) Preparar X/y ----
drop_cols = [TARGET_COL, CURVE_COL, "t_index"]

X_train_full = train_df.drop(columns=drop_cols, errors="ignore")
y_train_full = train_df[TARGET_COL].astype(float)

X_test_full  = test_df.drop(columns=drop_cols, errors="ignore")
y_test_full  = test_df[TARGET_COL].astype(float)

# ---- (3) Pré-processamento ----
num_cols = X_train_full.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train_full.select_dtypes(exclude=[np.number]).columns.tolist()

if len(cat_cols) > 0:
    print("⚠️ Existem colunas categóricas:", cat_cols)
    print("Este pipeline atual usa só numéricas. Se precisar, eu adapto para OneHot.")

pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols)
    ],
    remainder="drop"
)

# ---- (4) XGBoost (melhores params do Optuna) ----
best_params = {
    "n_estimators": 800,
    "max_depth": 9,
    "learning_rate": 0.16920306292293242,
    "subsample": 0.9819393281062181,
    "colsample_bytree": 0.9288186113582297,
    "min_child_weight": 7.029942756969786,
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": RANDOM_STATE
}

best_xgb = XGBRegressor(**best_params)

pipe_best = Pipeline([
    ("pre", pre),
    ("model", best_xgb)
])

# ---- (5) Treinar e prever (curva inteira) ----
pipe_best.fit(X_train_full, y_train_full)
y_pred = pipe_best.predict(X_test_full)

# Métricas
mse  = mean_squared_error(y_test_full, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_full, y_pred)
r2   = r2_score(y_test_full, y_pred)

print("\n=== XGBOOST - MÉTRICAS NO TESTE (curva inteira) ===")
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)

# ---- (6) Salvar arquivo com predições ----
out_pred = test_df.copy()
out_pred["R_pred"] = y_pred
out_pred["erro"] = out_pred[TARGET_COL].astype(float) - out_pred["R_pred"]
out_pred.to_csv("/content/predicoes_teste.csv", index=False)

print("\n✅ Arquivo salvo: /content/predicoes_teste.csv")

# ---- (7) Plot curva real vs predita (1 curva) ----
curve_example = test_df[CURVE_COL].iloc[0]
mask = (test_df[CURVE_COL] == curve_example)

t = test_df.loc[mask, TIME_COL].values
y_real = y_test_full.loc[mask].values
y_hat  = y_pred[mask.values]

order = np.argsort(t)

plt.figure()
plt.plot(t[order], y_real[order], label="Real")
plt.plot(t[order], y_hat[order], label="Predito")
plt.xlabel("Tempo (Q_time)")
plt.ylabel("Pressão (R_pressure)")
plt.title(f"Curva real vs predita — curve_id={curve_example}")
plt.legend()
plt.show()

# ---- (8) Plot R² acumulado ao longo do tempo (global) ----
order_global = np.argsort(test_df[TIME_COL].values)

y_real_ord = y_test_full.values[order_global]
y_pred_ord = y_pred[order_global]

r2_progressivo = []
idxs = list(range(50, len(y_real_ord), 200))

for i in idxs:
    r2_progressivo.append(r2_score(y_real_ord[:i], y_pred_ord[:i]))

plt.figure()
plt.plot(range(len(r2_progressivo)), r2_progressivo)
plt.xlabel("Blocos de tempo acumulados")
plt.ylabel("R² acumulado")
plt.title("Evolução do R² ao longo do tempo (teste)")
plt.show()

# ---- (9) Erro vs variáveis (2 a 2) ----
analysis_df = X_test_full.copy()
analysis_df["erro"] = out_pred["erro"].values

for col in num_cols:
    plt.figure()
    plt.scatter(analysis_df[col], analysis_df["erro"], s=5)
    plt.xlabel(col)
    plt.ylabel("Erro (Real - Predito)")
    plt.title(f"Erro vs {col}")
    plt.show()

# =========================================================
# (10) TABPFN em amostra (1000) + XGB na mesma amostra
# =========================================================
from tabpfn import TabPFNRegressor

X_all = X_train_full.copy()
y_all = y_train_full.copy()

SAMPLE_N = 1000
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    len(X_all), size=min(SAMPLE_N, len(X_all)), replace=False
)

X_s = X_all.iloc[sample_idx].copy()
y_s = y_all.iloc[sample_idx].copy()

X_tr, X_va, y_tr, y_va = train_test_split(
    X_s, y_s, test_size=0.2, random_state=RANDOM_STATE
)

# Imputer (TabPFN precisa sem NaN)
imp = SimpleImputer(strategy="median")
X_tr_imp = imp.fit_transform(X_tr)
X_va_imp = imp.transform(X_va)

# --- TABPFN ---
tab_model = TabPFNRegressor(device="cpu")
tab_model.fit(X_tr_imp, y_tr)
y_hat_tab = tab_model.predict(X_va_imp)

rmse_tab = np.sqrt(mean_squared_error(y_va, y_hat_tab))
r2_tab   = r2_score(y_va, y_hat_tab)

# --- XGB na mesma amostra (para comparar justo) ---
xgb_small = Pipeline([
    ("pre", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])),
    ("model", XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE
    ))
])

xgb_small.fit(X_tr, y_tr)
y_hat_xgb = xgb_small.predict(X_va)

rmse_xgb = np.sqrt(mean_squared_error(y_va, y_hat_xgb))
r2_xgb   = r2_score(y_va, y_hat_xgb)

comp = pd.DataFrame({
    "Modelo": ["XGBoost (amostra)", "TabPFN (amostra)"],
    "RMSE": [rmse_xgb, rmse_tab],
    "R2": [r2_xgb, r2_tab]
})

print("\n=== COMPARAÇÃO JUSTA NA MESMA AMOSTRA (1000) ===")
print(comp)